# Project Master Runner

This is the central entry point for the **Ecommerce Sentiment Analysis** project. It imports the logic from internal notebooks and executes the analysis pipeline using TensorFlow Metal acceleration.

## 1. Data Exploration & Cleaning

In [ ]:
%load_ext autoreload
%autoreload 2

# Import the DataExploration class from the notebooks folder
import sys
import os
import pandas as pd
sys.path.append(os.path.abspath('notebooks'))

from notebooks.CapstoneData import DataExploration

# Define the dataset path
data_path = 'Ecommerce_dataset/train_data.csv'

# Instantiate and run the analysis
eda = DataExploration(data_path)
eda.get_summary()
eda.get_sentiment_distribution()
eda.remove_nulls(columns=['reviews.text', 'sentiment'])

print(f"\nCleaned data ready with {len(eda.df)} rows.")

## 2. Model Training

Run the cell below to train all 6 models using the cleaned `eda.df` DataFrame. 

> **Note:** Deep learning models (BERT/DistilBERT/RoBERTa/RNN) will use the Metal GPU on your Mac for faster training.

In [32]:
import sys
import os
import tensorflow as tf
import importlib

# GPU setup
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
print("GPUs available for training:", gpus)

sys.path.append(os.path.abspath('notebooks'))

from notebooks.sentiment_analysis_distilbert import SentimentAnalysisDistilBERT
from notebooks.sentiment_analysis_BERT import SentimentAnalysisBERT
from notebooks.sentiment_analysis_RNN import SentimentAnalysisRNN
from notebooks.sentiment_analysis_Logistic import SentimentAnalysis
from notebooks.sentiment_analysis_SVM import SentimentAnalysisSVM
from notebooks.sentiment_analysis_Roberta import SentimentAnalysisHF

def train_all_models(df):
    print("--- Starting Training Pipeline using eda.df ---\n")
    results = {}
    
    # 1. Logistic Regression
    print("Training Logistic Regression...")
    lr = SentimentAnalysis()
    results['Logistic'] = lr.train(df)
    lr.persistModel('sentiment_model_TFD_LR.pkl')
    
    # 2. SVM
    print("\nTraining SVM...")
    svm = SentimentAnalysisSVM()
    results['SVM'] = svm.train(df)
    svm.persistModel('sentiment_model_SVM.pkl')
    
    # 3. RNN
    print("\nTraining RNN (5 epochs)... ")
    rnn = SentimentAnalysisRNN()
    results['RNN'] = rnn.train(df, epochs=5)
    rnn.persistModel('sentiment_model_RNN.pkl')
    
    # 4. DistilBERT
    print("\nTraining DistilBERT (1 epoch)... ")
    db = SentimentAnalysisDistilBERT()
    results['DistilBERT'] = db.train(df, epochs=1)
    db.persistModel('sentiment_model_distilbert.pkl')
    
    # 5. BERT
    print("\nTraining BERT (1 epoch)... ")
    bert = SentimentAnalysisBERT()
    results['BERT'] = bert.train(df, epochs=1)
    bert.persistModel('sentiment_model_BERT.pkl')
    
    # 6. RoBERTa
    print("\nSaving RoBERTa Metadata...")
    rob = SentimentAnalysisHF()
    rob.persistModel('sentiment_model_roberta.pkl')
    results['RoBERTa'] = "Success (Pre-trained)"
    
    print("\n✅ ALL MODELS TRAINED AND SAVED")
    return results

# RUN TRAINING
results = train_all_models(eda.df)

GPUs available for training: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
--- Starting Training Pipeline using eda.df ---

Training Logistic Regression...


TypeError: stat: path should be string, bytes, os.PathLike or integer, not DataFrame

## 3. Sentiment Models Initialization & Comparison

Initialize all models from their saved files to verify they load correctly.

In [31]:
import sys
import os
sys.path.append(os.path.abspath('notebooks'))

from notebooks.sentiment_analysis_distilbert import SentimentAnalysisDistilBERT
from notebooks.sentiment_analysis_BERT import SentimentAnalysisBERT
from notebooks.sentiment_analysis_RNN import SentimentAnalysisRNN
from notebooks.sentiment_analysis_Logistic import SentimentAnalysis
from notebooks.sentiment_analysis_SVM import SentimentAnalysisSVM
from notebooks.sentiment_analysis_Roberta import SentimentAnalysisHF

# Model Paths
paths = {
    'DistilBERT': 'sentiment_model_distilbert.pkl',
    'BERT':       'sentiment_model_BERT.pkl',
    'RNN':        'sentiment_model_RNN.pkl',
    'Logistic':   'sentiment_model_TFD_LR.pkl',
    'SVM':        'sentiment_model_SVM.pkl',
    'RoBERTa':    'sentiment_model_roberta.pkl'
}

models = {}

for name, path in paths.items():
    if os.path.exists(path):
        print(f"Loading {name}...")
        if name == 'DistilBERT': models[name] = SentimentAnalysisDistilBERT(model_path=path)
        elif name == 'BERT':     models[name] = SentimentAnalysisBERT(model_path=path)
        elif name == 'RNN':      models[name] = SentimentAnalysisRNN(model_path=path)
        elif name == 'Logistic': models[name] = SentimentAnalysis(model_path=path)
        elif name == 'SVM':      models[name] = SentimentAnalysisSVM(model_path=path)
        elif name == 'RoBERTa':  models[name] = SentimentAnalysisHF(model_path=path)
    else:
        print(f"❌ {name} not found at {path}")

print(f"\nInitialized {len(models)} models: {list(models.keys())}")

❌ DistilBERT not found at sentiment_model_distilbert.pkl
❌ BERT not found at sentiment_model_BERT.pkl
❌ RNN not found at sentiment_model_RNN.pkl
❌ Logistic not found at sentiment_model_TFD_LR.pkl
❌ SVM not found at sentiment_model_SVM.pkl
❌ RoBERTa not found at sentiment_model_roberta.pkl

Initialized 0 models: []


## 4. Sample Prediction

Testing the models with a sample review.

In [ ]:
sample_text = "This product is absolutely amazing, I love it!"
sample_title = "Great purchase"

if not models:
    print("No models loaded. Please train them in Section 2 first.")
else:
    print(f"Review: {sample_text}\n")
    for name, model in models.items():
        try:
            prediction = model.predict(sample_text, sample_title)
            print(f"[{name:10}] Prediction: {prediction}")
        except Exception as e:
            print(f"[{name:10}] Error: {e}")

## 5. Visual Analytics Report

Generating detailed reports using the SentimentReporter.

In [ ]:
from notebooks.sentiment_reporter import SentimentReporter
reporter = SentimentReporter(output_dir='personal_update/outputs')

print("Generating Visual Analytics Report...")
reporter.generate_visualizations(eda.df)

print("\n✅ PIPELINE COMPLETE")
print("Reports saved in 'personal_update/outputs/'")